# Полный пайплайн v5: CatBoost + XGBoost + ZILN → Ensemble

### Этап A: CatBoost seed-avg
### Этап B: XGBoost
### Этап C: ZILN MLP
### Этап D: Blendings

In [2]:
import polars as pl
import pandas as pd
import numpy as np
import time
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from catboost import CatBoostRegressor, Pool
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, QuantileTransformer

FEATURES_DIR = Path("../data/processed/features_v6")
PROCESSED = Path("../data/processed")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU: {free/1024**2:.0f}MB free / {total/1024**2:.0f}MB total")

def rmsle_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(np.log1p(np.clip(y_true, 0, None)),
                                      np.log1p(np.clip(y_pred, 0, None))))

def load_fold(fold_path: Path) -> pd.DataFrame:
    return pl.read_parquet(fold_path / "batch_*.parquet").to_pandas()

print("\nЗагрузка фолдов v6...")
fold_dirs = sorted(FEATURES_DIR.glob("fold_[0-9]*"))
N_FOLDS = len(fold_dirs)
all_folds = []
for fd in fold_dirs:
    df = load_fold(fd)
    all_folds.append(df)
    n_buy = (df["target"] > 0).sum()
    print(f"  {fd.name}: {len(df):,} | buyers={n_buy:,} ({n_buy/len(df)*100:.1f}%)")

test_df = load_fold(FEATURES_DIR / "fold_test")
print(f"  fold_test: {len(test_df):,}")

drop_cols = ["user_id", "anchor_date", "target"]
features = [c for c in test_df.columns if c not in drop_cols]
print(f"\nФичей: {len(features)} | Фолдов: {N_FOLDS} | Device: {DEVICE}")

GPU: 5073MB free / 6140MB total

Загрузка фолдов v6...
  fold_00: 250,000 | buyers=141,633 (56.7%)
  fold_01: 250,000 | buyers=143,878 (57.6%)
  fold_02: 250,000 | buyers=139,924 (56.0%)
  fold_03: 250,000 | buyers=133,886 (53.6%)
  fold_04: 250,000 | buyers=135,165 (54.1%)
  fold_test: 250,000

Фичей: 437 | Фолдов: 5 | Device: cuda


---
## Этап A: CatBoost Seed Averaging

In [2]:
SEEDS = [42, 123, 777, 2024, 31415]
CB_PARAMS = {
    'iterations': 2000,
    'learning_rate': 0.03,
    'depth': 6,
    'loss_function': 'RMSE', 
    'eval_metric': 'RMSE',
    'task_type': 'GPU', 
    'devices': '0',
    'od_type': 'Iter', 
    'early_stopping_rounds': 150, 
    'verbose': False,
}

print("=== Этап A: CatBoost Seed Avg ===")
t0 = time.time()

oof_cb = np.zeros(len(all_folds[-1]))
test_preds_cb_all_seeds = []
fold_scores_cb = []

for seed in SEEDS:
    params = {**CB_PARAMS, 'random_seed': seed}
    seed_scores = []

    for val_idx in range(1, N_FOLDS):
        train_df = all_folds[val_idx - 1]
        val_df = all_folds[val_idx]

        X_tr = train_df[features]
        y_tr = np.log1p(np.clip(train_df["target"], 0, None))
        X_vl = val_df[features]
        y_vl = val_df["target"].values
        y_vl_log = np.log1p(np.clip(y_vl, 0, None))

        model = CatBoostRegressor(**params)
        model.fit(Pool(X_tr, y_tr), eval_set=Pool(X_vl, y_vl_log))

        val_pred = np.expm1(np.clip(model.predict(X_vl), 0, None))
        score = rmsle_score(y_vl, val_pred)
        seed_scores.append(score)

        if val_idx == N_FOLDS - 1:
            oof_cb += val_pred / len(SEEDS)

    model_final = CatBoostRegressor(**params)
    model_final.fit(
        Pool(all_folds[-1][features], np.log1p(np.clip(all_folds[-1]["target"], 0, None))),
        eval_set=Pool(all_folds[-2][features], np.log1p(np.clip(all_folds[-2]["target"].values, 0, None)))
    )
    test_pred = np.expm1(np.clip(model_final.predict(test_df[features]), 0, None))
    test_preds_cb_all_seeds.append(test_pred)

    mean_score = np.mean(seed_scores)
    fold_scores_cb.append(mean_score)
    print(f"  Seed {seed}: mean CV={mean_score:.5f} | per-fold={[f'{s:.5f}' for s in seed_scores]} | {time.time()-t0:.0f}s")

test_preds_cb = np.mean(test_preds_cb_all_seeds, axis=0)
oof_score_cb = rmsle_score(all_folds[-1]["target"].values, oof_cb)
print(f"\n  CatBoost OOF RMSLE: {oof_score_cb:.5f}")

np.save(PROCESSED / "oof_v6_catboost.npy", oof_cb)
sub_cb = pd.DataFrame({"user_id": test_df["user_id"], "predict": np.clip(test_preds_cb, 0, None)})
sub_cb.to_csv(PROCESSED / "catboost_v6_seed_avg.csv", index=False)
print(f"  Saved: catboost_v6_seed_avg.csv (mean={sub_cb['predict'].mean():.2f})")

del model, model_final
gc.collect()
torch.cuda.empty_cache()

=== Этап A: CatBoost Seed Avg ===
  Seed 42: mean CV=1.71066 | per-fold=['1.74482', '1.73838', '1.69773', '1.66171'] | 132s
  Seed 123: mean CV=1.71058 | per-fold=['1.74468', '1.73832', '1.69751', '1.66180'] | 275s
  Seed 777: mean CV=1.71064 | per-fold=['1.74429', '1.73856', '1.69803', '1.66170'] | 414s
  Seed 2024: mean CV=1.71060 | per-fold=['1.74430', '1.73831', '1.69785', '1.66193'] | 550s
  Seed 31415: mean CV=1.71053 | per-fold=['1.74451', '1.73816', '1.69790', '1.66155'] | 681s

  CatBoost OOF RMSLE: 1.66115
  Saved: catboost_v6_seed_avg.csv (mean=35.62)


---
## Этап B: XGBoost

In [3]:
print("=== Этап B: XGBoost ===")
t0 = time.time()

XGB_PARAMS = {
    'objective': 'reg:squarederror', 'eval_metric': 'rmse',
    'learning_rate': 0.03, 'max_depth': 6,
    'subsample': 0.8, 'colsample_bytree': 0.7,
    'reg_lambda': 5.0, 'reg_alpha': 2.0,
    'tree_method': 'hist', 'device': 'cuda', 'random_state': 42
}

oof_xgb = np.zeros(len(all_folds[-1]))
xgb_scores = []

for val_idx in range(1, N_FOLDS):
    train_df = all_folds[val_idx - 1]
    val_df = all_folds[val_idx]

    y_tr_log = np.log1p(np.clip(train_df["target"].values, 0, None))
    y_vl = val_df["target"].values
    y_vl_log = np.log1p(np.clip(y_vl, 0, None))

    dtrain = xgb.DMatrix(train_df[features], label=y_tr_log)
    dval = xgb.DMatrix(val_df[features], label=y_vl_log)

    model = xgb.train(XGB_PARAMS, dtrain, num_boost_round=3000,
                      evals=[(dval, 'val')], early_stopping_rounds=150, verbose_eval=False)

    val_pred = np.expm1(np.clip(model.predict(dval), 0, None))
    score = rmsle_score(y_vl, val_pred)
    xgb_scores.append(score)
    print(f"  fold {val_idx}: RMSLE={score:.5f}")

    if val_idx == N_FOLDS - 1:
        oof_xgb = val_pred

dtrain_final = xgb.DMatrix(all_folds[-1][features], label=np.log1p(np.clip(all_folds[-1]["target"].values, 0, None)))
deval_final = xgb.DMatrix(all_folds[-2][features], label=np.log1p(np.clip(all_folds[-2]["target"].values, 0, None)))
dtest = xgb.DMatrix(test_df[features])

model = xgb.train(XGB_PARAMS, dtrain_final, num_boost_round=3000,
                  evals=[(deval_final, 'val')], early_stopping_rounds=150, verbose_eval=False)
test_preds_xgb = np.expm1(np.clip(model.predict(dtest), 0, None))

oof_score_xgb = rmsle_score(all_folds[-1]["target"].values, oof_xgb)
print(f"\n  XGBoost OOF RMSLE: {oof_score_xgb:.5f}")

np.save(PROCESSED / "oof_v6_xgboost.npy", oof_xgb)
sub_xgb = pd.DataFrame({"user_id": test_df["user_id"], "predict": np.clip(test_preds_xgb, 0, None)})
sub_xgb.to_csv(PROCESSED / "xgboost_v6_submission.csv", index=False)
print(f"  Saved: xgboost_v6_submission.csv (mean={sub_xgb['predict'].mean():.2f})")

del model, dtrain, dval, dtrain_final, deval_final, dtest
gc.collect()
torch.cuda.empty_cache()
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU после очистки: {free/1024**2:.0f}MB free")

=== Этап B: XGBoost ===
  fold 1: RMSLE=1.72224
  fold 2: RMSLE=1.71486
  fold 3: RMSLE=1.67945
  fold 4: RMSLE=1.64183

  XGBoost OOF RMSLE: 1.64183
  Saved: xgboost_v6_submission.csv (mean=36.37)
GPU после очистки: 5057MB free


---
## Этап C: ZILN MLP

In [3]:
class TabularMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(0.4),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.BatchNorm1d(64), nn.GELU(), nn.Dropout(0.1),
            nn.Linear(64, 3)
        )
    def forward(self, x):
        return self.net(x)

class ZILNLoss(nn.Module):
    """Zero-Inflated LogNormal loss: BCE для classification + LogNormal NLL для regression."""
    def forward(self, preds, target):
        p_logits = preds[:, 0]
        mu = preds[:, 1]
        rho = torch.clamp(preds[:, 2], -10, 10)
        sigma = F.softplus(rho) + 1e-6
        is_zero = (target == 0).float()
        is_positive = (target > 0).float()
        cls_loss = F.binary_cross_entropy_with_logits(p_logits, is_positive, reduction='none')
        log_target = torch.log(target + 1e-6)
        reg_loss = (0.5 * torch.log(2 * torch.pi * sigma**2) +
                   ((log_target - mu)**2) / (2 * sigma**2) + log_target)
        return torch.mean(is_zero * cls_loss + is_positive * (cls_loss + reg_loss))

def extract_predictions(preds, mode="median"):
    """Три режима предсказания:
    - median: p * exp(mu)              — медиана LogNormal, стабильная
    - mean:   p * exp(mu + sigma^2/2)  — среднее LogNormal, завышает
    - v1:     expm1(p * mu)            — оригинальная формула (работала на v4)
    """
    p = torch.sigmoid(preds[:, 0])
    mu = torch.clamp(preds[:, 1], -5, 12)
    
    if mode == "median":
        return torch.clamp(p * torch.exp(mu), 0, 1e6)
    elif mode == "mean":
        sigma = F.softplus(torch.clamp(preds[:, 2], -10, 3)) + 1e-6
        log_exp = torch.clamp(mu + 0.5 * sigma**2, -5, 14)
        return torch.clamp(p * torch.exp(log_exp), 0, 1e6)
    else:
        return torch.clamp(torch.expm1(p * mu), 0, 1e6)

class LTVDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32) if y is not None else None
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        return (self.X[idx], self.y[idx]) if self.y is not None else self.X[idx]

print("=== Этап C: ZILN MLP v3 ===")

train_z = all_folds[-2]
val_z = all_folds[-1]

scaler = QuantileTransformer(n_quantiles=1000, output_distribution='normal', random_state=42)

X_train_s = scaler.fit_transform(train_z[features].fillna(0).values)
X_val_s = scaler.transform(val_z[features].fillna(0).values)
X_test_s = scaler.transform(test_df[features].fillna(0).values)

X_train_s = np.clip(X_train_s, -5.0, 5.0)
X_val_s = np.clip(X_val_s, -5.0, 5.0)
X_test_s = np.clip(X_test_s, -5.0, 5.0)

y_train_raw = np.clip(train_z["target"].values, 0, None).astype(np.float32)
y_val_raw = np.clip(val_z["target"].values, 0, None).astype(np.float32)

print(f"NaN: {np.isnan(X_train_s).sum()}, Inf: {np.isinf(X_train_s).sum()}, Max: {np.abs(X_train_s).max():.1f}")
print(f"Target: mean={y_train_raw.mean():.1f}, zeros={100*(y_train_raw==0).mean():.1f}%")

train_loader = DataLoader(LTVDataset(X_train_s, y_train_raw), batch_size=256, shuffle=True)
val_loader = DataLoader(LTVDataset(X_val_s, y_val_raw), batch_size=256, shuffle=False)
test_loader = DataLoader(LTVDataset(X_test_s), batch_size=256, shuffle=False)

LR = 3e-4
EPOCHS = 50
PATIENCE = 15
WARMUP = 5
PRED_MODE = "v1"

model_ziln = TabularMLP(input_dim=len(features)).to(DEVICE)
optimizer = torch.optim.AdamW(model_ziln.parameters(), lr=LR, weight_decay=1e-3)
criterion = ZILNLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4)

best_rmsle = float('inf')
best_state = None
patience_counter = 0

for epoch in range(EPOCHS):
    model_ziln.train()
    train_loss = 0
    t0 = time.time()
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model_ziln(X_b), y_b)
        if torch.isnan(loss):
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_ziln.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item() * X_b.size(0)
    train_loss /= len(train_loader.dataset)

    model_ziln.eval()
    val_preds = []
    with torch.no_grad():
        for X_b, _ in val_loader:
            preds = model_ziln(X_b.to(DEVICE))
            val_preds.extend(extract_predictions(preds, mode=PRED_MODE).cpu().numpy())
    val_arr = np.array(val_preds)
    val_rmsle = rmsle_score(y_val_raw, val_arr)
    scheduler.step(val_rmsle)

    lr_now = optimizer.param_groups[0]['lr']
    print(f"  Epoch {epoch+1:02d}/{EPOCHS} | Loss: {train_loss:.4f} | RMSLE: {val_rmsle:.5f} | LR: {lr_now:.1e} | mean: {val_arr.mean():.1f} | {time.time()-t0:.1f}s")

    if val_rmsle < best_rmsle:
        best_rmsle = val_rmsle
        best_state = {k: v.clone() for k, v in model_ziln.state_dict().items()}
        patience_counter = 0
    elif epoch >= WARMUP:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1} (best: {best_rmsle:.5f})")
            break

print(f"\n  Best RMSLE ({PRED_MODE}): {best_rmsle:.5f}")

=== Этап C: ZILN MLP v3 ===
NaN: 0, Inf: 0, Max: 5.0
Target: mean=83.2, zeros=46.4%
  Epoch 01/50 | Loss: 4.4021 | RMSLE: 1.68070 | LR: 3.0e-04 | mean: 31.2 | 27.4s
  Epoch 02/50 | Loss: 3.5617 | RMSLE: 1.67775 | LR: 3.0e-04 | mean: 31.2 | 10.8s
  Epoch 03/50 | Loss: 3.5548 | RMSLE: 1.67975 | LR: 3.0e-04 | mean: 27.8 | 9.5s
  Epoch 04/50 | Loss: 3.5497 | RMSLE: 1.67544 | LR: 3.0e-04 | mean: 31.1 | 10.6s
  Epoch 05/50 | Loss: 3.5473 | RMSLE: 1.67488 | LR: 3.0e-04 | mean: 30.6 | 13.5s
  Epoch 06/50 | Loss: 3.5453 | RMSLE: 1.67434 | LR: 3.0e-04 | mean: 31.8 | 10.8s
  Epoch 07/50 | Loss: 3.5436 | RMSLE: 1.67303 | LR: 3.0e-04 | mean: 33.2 | 13.5s
  Epoch 08/50 | Loss: 3.5417 | RMSLE: 1.67649 | LR: 3.0e-04 | mean: 28.9 | 12.0s
  Epoch 09/50 | Loss: 3.5403 | RMSLE: 1.67529 | LR: 3.0e-04 | mean: 29.3 | 11.0s
  Epoch 10/50 | Loss: 3.5394 | RMSLE: 1.67384 | LR: 3.0e-04 | mean: 31.6 | 11.9s
  Epoch 11/50 | Loss: 3.5382 | RMSLE: 1.67336 | LR: 3.0e-04 | mean: 34.6 | 19.1s
  Epoch 12/50 | Loss: 3.53

In [4]:
model_ziln.load_state_dict(best_state)
model_ziln.eval()

for mode in ["median", "v1", "mean"]:
    oof_preds = []
    with torch.no_grad():
        for X_b, _ in val_loader:
            oof_preds.extend(extract_predictions(model_ziln(X_b.to(DEVICE)), mode=mode).cpu().numpy())
    oof_arr = np.array(oof_preds)
    oof_rmsle = rmsle_score(y_val_raw, oof_arr)
    
    test_preds = []
    with torch.no_grad():
        for X_b in test_loader:
            test_preds.extend(extract_predictions(model_ziln(X_b.to(DEVICE)), mode=mode).cpu().numpy())
    test_arr = np.array(test_preds)
    
    print(f"  {mode:>7s}: OOF RMSLE={oof_rmsle:.5f} | val_mean={oof_arr.mean():.1f} | test_mean={test_arr.mean():.1f}")

    if mode == "v1": 
        np.save(PROCESSED / "oof_v6_ziln.npy", oof_arr)
        sub = pd.DataFrame({"user_id": test_df["user_id"], "predict": np.clip(test_arr, 0, None)})
        sub.to_csv(PROCESSED / "ziln_v6_v1_submission.csv", index=False)

print(f"\n  Saved: ziln_v6_v1_submission.csv")

   median: OOF RMSLE=1.92206 | val_mean=47.9 | test_mean=47.7
       v1: OOF RMSLE=1.66912 | val_mean=34.4 | test_mean=34.4
     mean: OOF RMSLE=2.31845 | val_mean=91.6 | test_mean=91.1

  Saved: ziln_v6_v1_submission.csv


---
## Этап D: бленды

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED = Path("../data/processed")

cb_df = pd.read_csv(PROCESSED / "catboost_v6_seed_avg.csv").sort_values("user_id")
xgb_df = pd.read_csv(PROCESSED / "xgboost_v6_submission.csv").sort_values("user_id")
mlp_df = pd.read_csv(PROCESSED / "ziln_v6_v1_submission.csv").sort_values("user_id")

test_cb = cb_df["predict"].values
test_xgb = xgb_df["predict"].values
test_mlp = mlp_df["predict"].values
user_ids = cb_df["user_id"].values

print("=== Поиск идеального базового ансамбля (Чистые веса) ===")


blends = {
    "v6_arithmetic_cb70_mlp20_xgb10.csv": 0.70 * test_cb + 0.20 * test_mlp + 0.10 * test_xgb,
}

for name, preds in blends.items():
    pd.DataFrame({"user_id": user_ids, "predict": np.clip(preds, 0, None)}).to_csv(PROCESSED / name, index=False)
    print(f"Готов: {name} | mean = {preds.mean():.2f}")

=== Поиск идеального базового ансамбля (Чистые веса) ===
Готов: v6_arithmetic_cb70_mlp20_xgb10.csv | mean = 35.44
